In [1]:
import random
import tensorflow as tf

In [2]:
def generate_regression_data(w, b, noise=0.01, num_train=1000, num_val=1000):
        n = num_train + num_val
        X = tf.random.normal((n, w.shape[0]))
        noise = tf.random.normal((n, 1)) * noise
        y = tf.matmul(X, tf.reshape(w, (-1, 1))) + b + noise
        return X, y

In [3]:
X, y = generate_regression_data(tf.constant([2.0, -3.4]), 4.2)

In [4]:
X[0], y[0]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 0.05571058, -1.0649918 ], dtype=float32)>,
 <tf.Tensor: shape=(1,), dtype=float32, numpy=array([7.935063], dtype=float32)>)

In [36]:
test_data = tf.data.Dataset.from_tensors((X, y))


In [37]:
list(test_data.as_numpy_iterator())

[(array([[ 0.05571058, -1.0649918 ],
         [-0.27138785, -1.2341973 ],
         [ 0.03866617,  2.1108649 ],
         ...,
         [-0.8164025 , -1.6101928 ],
         [ 0.9521109 ,  0.79359734],
         [-0.7481617 , -2.0486593 ]], shape=(2000, 2), dtype=float32),
  array([[ 7.935063 ],
         [ 7.843923 ],
         [-2.8968074],
         ...,
         [ 8.029936 ],
         [ 3.4078236],
         [ 9.642019 ]], shape=(2000, 1), dtype=float32))]

In [41]:
test_batch = test_data.shuffle(1000).batch(32)

In [8]:
class LinearRegressionModel(tf.Module):
    def __init__(self):
        super().__init__()
        self.w = tf.Variable(tf.random.normal((2, 1)), name='w', trainable=True)
        self.b = tf.Variable(tf.zeros(1), name='b', trainable=True)

    def __call__(self, x):
        return tf.matmul(x, self.w) + self.b
    

In [9]:
def loss_fn(y_pred, y_true):
    return tf.reduce_mean(tf.square(y_pred - y_true) / 2)

In [10]:
module = LinearRegressionModel()

In [11]:
loss_fn(module(X), y)

<tf.Tensor: shape=(), dtype=float32, numpy=9.719066619873047>

In [12]:
# Given a callable model, inputs, outputs, and a learning rate...
def train(model, x, y, learning_rate):

  with tf.GradientTape() as t:
    # Trainable variables are automatically tracked by GradientTape
    current_loss = loss_fn(model(x), y)

  # Use GradientTape to calculate the gradients with respect to W and b
  dw, db = t.gradient(current_loss, [model.w, model.b])

  # Subtract the gradient scaled by the learning rate
  model.w.assign_sub(learning_rate * dw)
  model.b.assign_sub(learning_rate * db)

In [21]:
def training_loop(model, x, y):

  for epoch in range(20):
    # Update the model with the single giant batch
    train(model, x, y, learning_rate=0.1)
    current_loss = loss_fn(y, model(x))

    print(f"Epoch {epoch:2d}:")
    print("Loss: {:0.3f}   ".format(current_loss))

In [22]:
training_loop(module, X, y)

Epoch  0:
Loss: 0.511   
Epoch  1:
Loss: 0.415   
Epoch  2:
Loss: 0.336   
Epoch  3:
Loss: 0.273   
Epoch  4:
Loss: 0.221   
Epoch  5:
Loss: 0.179   
Epoch  6:
Loss: 0.145   
Epoch  7:
Loss: 0.118   
Epoch  8:
Loss: 0.096   
Epoch  9:
Loss: 0.078   
Epoch 10:
Loss: 0.063   
Epoch 11:
Loss: 0.051   
Epoch 12:
Loss: 0.041   
Epoch 13:
Loss: 0.034   
Epoch 14:
Loss: 0.027   
Epoch 15:
Loss: 0.022   
Epoch 16:
Loss: 0.018   
Epoch 17:
Loss: 0.015   
Epoch 18:
Loss: 0.012   
Epoch 19:
Loss: 0.010   


In [47]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(1)
    ])

model.compile(optimizer='SGD',
              loss=tf.keras.losses.MeanSquaredError())

model.fit(test_batch, epochs=300)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - loss: 40.4863
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 38.9465
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 37.4652
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 36.0403
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 34.6696
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 33.3511
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 32.0827
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 30.8626
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 29.6889
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 28.5598
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 27.4738
Epoch 12/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 26.4290
Epoch 13/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 25.4240
Epoch 14/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 24.4572
Epoch 15/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 23.5272
Epo

In [48]:
model.get_weights()

[array([[ 1.9977485],
        [-3.3853648]], dtype=float32),
 array([4.1871657], dtype=float32)]